# Notebook 06 — Optimisation des Paramètres

**Objectif :** Améliorer le MRR baseline de 0.2745 (SentenceTransformers, notebook 05) par tuning systématique.

**Ce qu'on optimise :**
1. Stratégie de construction de requête (5 variantes)
2. Paramètres BM25+ (k1, b, delta) — grid search
3. Tokenisation BM25 (split simple vs NLTK vs stemming)

**Ce qu'on ne change pas :** modèle d'embedding `all-MiniLM-L6-v2`, K=100 pour soumission.

## Cellule 1 — Imports et Chargement

In [ ]:
import json
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

DATA_DIR    = '../data'
MODELS_DIR  = '../models'
OUTPUTS_DIR = '../outputs'

# --- Corpus ---
df_docs = pd.read_pickle(os.path.join(DATA_DIR, 'df_docs_preprocessed.pkl'))

with open(os.path.join(DATA_DIR, 'queries_train.json'), 'r', encoding='utf-8') as f:
    queries_train = json.load(f)

with open(os.path.join(DATA_DIR, 'qgts_train.json'), 'r', encoding='utf-8') as f:
    qgts_train = json.load(f)

with open(os.path.join(DATA_DIR, 'queries_test.json'), 'r', encoding='utf-8') as f:
    queries_test = json.load(f)

id_to_idx = {doc_id: idx for idx, doc_id in enumerate(df_docs['id'])}
idx_to_id = {idx: doc_id for doc_id, idx in id_to_idx.items()}

# --- Embeddings pré-calculés ---
corpus_embeddings = np.load(os.path.join(MODELS_DIR, 'corpus_embeddings.npy'))

# --- Sentence Transformer ---
model_st = SentenceTransformer('all-MiniLM-L6-v2')

print(f'Corpus         : {len(df_docs):,} documents')
print(f'Embeddings     : {corpus_embeddings.shape}')
print(f'Queries train  : {len(queries_train)}')
print(f'Queries test   : {len(queries_test)}')
print('Chargement OK.')

## Cellule 2 — Fonctions d'Évaluation

In [2]:
def get_relevant_ids(qgts, query_id):
    if query_id not in qgts:
        return []
    return [e['doc_id'] for e in qgts[query_id]['relevant_doc_ids']]


def _recall_at_k(retrieved, relevant_ids, k):
    rel_set = set(id_to_idx[r] for r in relevant_ids if r in id_to_idx)
    ret_set = set(retrieved[:k])
    return len(ret_set & rel_set) / max(len(rel_set), 1)


def _precision_at_k(retrieved, relevant_ids, k):
    rel_set = set(id_to_idx[r] for r in relevant_ids if r in id_to_idx)
    ret_set = set(retrieved[:k])
    return len(ret_set & rel_set) / k


def _mrr(retrieved, relevant_ids):
    rel_set = set(id_to_idx[r] for r in relevant_ids if r in id_to_idx)
    for rank, idx in enumerate(retrieved, start=1):
        if idx in rel_set:
            return 1.0 / rank
    return 0.0


def evaluate(search_fn, queries, qgts, k=10):
    """
    Évalue search_fn sur toutes les requêtes qui ont un ground truth.

    Args:
        search_fn: callable(query_text, k) -> {'topk_indices': list}
        queries  : liste de dicts requêtes
        qgts     : dict ground truth
        k        : @k pour les métriques

    Returns:
        dict avec MRR, Recall@k, Precision@k (moyennes)
    """
    mrrs, recs, precs = [], [], []
    for q in queries:
        q_id  = q['id']
        rel   = get_relevant_ids(qgts, q_id)
        if not rel:
            continue
        q_text    = q['_query_text']  # pré-calculé avant appel
        result    = search_fn(q_text, k)
        retrieved = result['topk_indices']
        mrrs.append(_mrr(retrieved, rel))
        recs.append(_recall_at_k(retrieved, rel, k))
        precs.append(_precision_at_k(retrieved, rel, k))
    return {
        f'MRR@{k}':       round(float(np.mean(mrrs)),  4),
        f'Recall@{k}':    round(float(np.mean(recs)),  4),
        f'Precision@{k}': round(float(np.mean(precs)), 4),
        'n_queries':      len(mrrs),
    }


print('Fonctions d\'évaluation prêtes.')

Fonctions d'évaluation prêtes.


## Cellule 3 — Tuning : Stratégie de Construction de Requête

On teste 5 façons de construire le texte de la requête sur les **embeddings** (pas de rebuild nécessaire).

In [3]:
# --- Définition des 5 stratégies ---
QUERY_STRATEGIES = {
    'text':           lambda q: q.get('text', '').strip(),
    'text+title':     lambda q: ' '.join(p for p in [q.get('text', ''), q.get('title', '')] if p).strip(),
    'text+title+tags': lambda q: ' '.join(p for p in [
                                     q.get('text', ''), q.get('title', ''),
                                     ' '.join(q['tags']) if q.get('tags') else ''
                                 ] if p).strip(),
    'title+tags':     lambda q: ' '.join(p for p in [
                                     q.get('title', ''),
                                     ' '.join(q['tags']) if q.get('tags') else ''
                                 ] if p).strip(),
    'tags':           lambda q: ' '.join(q['tags']) if q.get('tags') else q.get('text', '').strip(),
}


# --- Fonction de recherche par embeddings ---
def search_emb(query_text, k=10):
    q_emb  = model_st.encode([query_text], convert_to_numpy=True)
    scores = cosine_similarity(q_emb, corpus_embeddings).flatten()
    top_k  = np.argsort(scores)[::-1][:k]
    return {'topk_indices': top_k.tolist(), 'topk_scores': scores[top_k].tolist()}


# --- Évaluation des stratégies ---
K_EVAL = 10
rows_strategy = []

for strat_name, build_fn in QUERY_STRATEGIES.items():
    # Injecter le texte pré-calculé dans chaque requête
    queries_annotated = []
    for q in queries_train:
        qc = dict(q)
        qc['_query_text'] = build_fn(q)
        queries_annotated.append(qc)

    metrics = evaluate(search_emb, queries_annotated, qgts_train, k=K_EVAL)
    rows_strategy.append({'Stratégie': strat_name, **metrics})
    print(f'  [{strat_name:20s}]  MRR@{K_EVAL}={metrics[f"MRR@{K_EVAL}"]:.4f}  '
          f'Recall@{K_EVAL}={metrics[f"Recall@{K_EVAL}"]:.4f}')

df_strategy = pd.DataFrame(rows_strategy).set_index('Stratégie')

print()
print('=== Tableau : Impact de la stratégie de requête (Embeddings) ===')
display(df_strategy.style
        .format('{:.4f}', subset=[c for c in df_strategy.columns if c != 'n_queries'])
        .highlight_max(axis=0, subset=[f'MRR@{K_EVAL}'], color='lightgreen'))

best_strategy_name = df_strategy[f'MRR@{K_EVAL}'].idxmax()
best_strategy_fn   = QUERY_STRATEGIES[best_strategy_name]
best_strategy_mrr  = df_strategy[f'MRR@{K_EVAL}'].max()
print(f'\nMeilleure stratégie : "{best_strategy_name}"  MRR@{K_EVAL}={best_strategy_mrr:.4f}')

  [text                ]  MRR@10=0.4635  Recall@10=0.3105
  [text+title          ]  MRR@10=0.4635  Recall@10=0.3105
  [text+title+tags     ]  MRR@10=0.2745  Recall@10=0.1959
  [title+tags          ]  MRR@10=0.0000  Recall@10=0.0000
  [tags                ]  MRR@10=0.0000  Recall@10=0.0000

=== Tableau : Impact de la stratégie de requête (Embeddings) ===


,MRR@10,Recall@10,Precision@10,n_queries
Stratégie,,,,
text,0.4635,0.3105,0.1832,327
text+title,0.4635,0.3105,0.1832,327
text+title+tags,0.2745,0.1959,0.1141,327
title+tags,0.0000,0.0000,0.0000,327
tags,0.0000,0.0000,0.0000,327



Meilleure stratégie : "text"  MRR@10=0.4635


## Cellule 4 — Résumé : Meilleure Configuration Globale

Tableau comparatif baseline (text+title+tags) vs meilleure stratégie trouvée en cellule 3.

In [ ]:
def build_query_default(q):
    parts = [q.get('text', ''), q.get('title', '')]
    if q.get('tags'):
        parts.append(' '.join(q['tags']))
    return ' '.join(p for p in parts if p).strip()

queries_default    = [dict(q, _query_text=build_query_default(q)) for q in queries_train]
queries_best_strat = [dict(q, _query_text=best_strategy_fn(q))    for q in queries_train]

baseline_emb = evaluate(search_emb, queries_default,    qgts_train, k=K_EVAL)
tuned_emb    = evaluate(search_emb, queries_best_strat, qgts_train, k=K_EVAL)

summary_rows = [
    {'Config': '[BASELINE] Emb — text+title+tags',       **{k: v for k, v in baseline_emb.items() if k != 'n_queries'}},
    {'Config': f'[TUNED]    Emb — {best_strategy_name}', **{k: v for k, v in tuned_emb.items()    if k != 'n_queries'}},
]

df_summary = pd.DataFrame(summary_rows).set_index('Config')

print('=== TABLEAU COMPARATIF FINAL ===')
display(df_summary.style
        .format('{:.4f}')
        .highlight_max(axis=0, color='lightgreen')
        .highlight_min(axis=0, color='#ffcccc')
        .set_caption('Baseline vs Tuned — Embeddings'))

winner_mrr   = tuned_emb[f'MRR@{K_EVAL}']
baseline_mrr = baseline_emb[f'MRR@{K_EVAL}']
gain         = winner_mrr - baseline_mrr

print()
print(f'Gagnant      : Emb — {best_strategy_name}')
print(f'MRR@{K_EVAL}     : {winner_mrr:.4f}')
print(f'Gain         : {gain:+.4f} ({gain/baseline_mrr*100:+.1f}%)')

final_search_fn  = search_emb
final_query_fn   = best_strategy_fn
final_model_name = f'Embeddings + stratégie={best_strategy_name}'

print(f'\nConfiguration retenue : {final_model_name}')

## Cellule 7 — Génération de la Soumission Optimisée (`92iemj_v7_tuned.csv`)

In [7]:
K_SUBMIT = 100

rows_submit = []
for q_entry in queries_test:
    q_id   = q_entry['id']
    q_text = final_query_fn(q_entry)
    q_cat  = q_entry.get('category', '?') or '?'

    result  = final_search_fn(q_text, K_SUBMIT)
    doc_ids = [idx_to_id[idx] for idx in result['topk_indices']]

    rows_submit.append({
        'query_id':         q_id,
        'relevant_doc_ids': json.dumps(doc_ids),
        'category':         q_cat,
    })

submission_path = os.path.join(OUTPUTS_DIR, '92iemj_v7_tuned.csv')
df_submit = pd.DataFrame(rows_submit)
df_submit.to_csv(submission_path, index=False)

print(f'Soumission sauvegardée : {submission_path}')
print(f'Lignes : {len(df_submit)} requêtes × top-{K_SUBMIT} documents')
print()

# Vérification du format
assert len(df_submit) == 141, f'ERREUR : attendu 141 lignes, obtenu {len(df_submit)}'
n_docs_per_row = df_submit['relevant_doc_ids'].apply(lambda x: len(json.loads(x)))
assert (n_docs_per_row == K_SUBMIT).all(), f'ERREUR : certaines lignes ont != {K_SUBMIT} doc_ids'
print('Vérifications OK : 141 requêtes × 100 doc_ids chacune.')
print()

print('=== Aperçu de la soumission ===')
display(df_submit.head(5))

print()
print(f'=== Configuration finale utilisée ===')
print(f'  Modèle       : {final_model_name}')
print(f'  Stratégie    : {best_strategy_name}')
print(f'  MRR@10 train : {winner_mrr:.4f}  (baseline : {baseline_mrr:.4f})')
print(f'  Fichier      : outputs/92iemj_v7_tuned.csv')
print('\nPrêt pour soumission Kaggle.')

Soumission sauvegardée : ../outputs\92iemj_v7_tuned.csv
Lignes : 141 requêtes × top-100 documents

Vérifications OK : 141 requêtes × 100 doc_ids chacune.

=== Aperçu de la soumission ===


,query_id,relevant_doc_ids,category
0,4ffe16bc-5235-418d-9bf3-22d1f2c5796e_145437,"[""c583f3cd-b1ca-4b74-ac3e-1c6771eb6a8e_131332""...",?
1,1bb2bb20-7f45-4dcf-a94a-420c454f87b8_56473,"[""edd58474-cef1-4d75-b184-99ee27def6a2_116280""...",?
2,6a9a342c-1275-4bb3-a818-8bcce53fac4f_34507,"[""43fa6e5a-6ad6-4701-ba44-68b513409ffc_17053"",...",?
3,cb216e47-add6-41fd-974a-39251e4df3aa_6777,"[""8d766d9a-99a4-427e-88a1-2ffcdf634811_32731"",...",?
4,14f1d3f5-8271-400e-9ef2-8319de25c9a1_200748,"[""59108dd2-5f5a-4add-b947-3303c8aaa891_123331""...",?



=== Configuration finale utilisée ===
  Modèle       : Embeddings + stratégie=text
  Stratégie    : text
  MRR@10 train : 0.4635  (baseline : 0.2745)
  Fichier      : outputs/92iemj_v7_tuned.csv

Prêt pour soumission Kaggle.
